In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("parasharmanas/movie-recommendation-system")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/movie-recommendation-system


In [3]:
! pip install fuzzywuzzy

In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import NearestNeighbors
from fuzzywuzzy import process

/usr/local/lib/python3.11/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [5]:
!ls -lah /root/.cache/kagglehub/datasets/parasharmanas/movie-recommendation-system/versions/1


total 650M
drwxr-xr-x 2 root root 4.0K Jun 24 01:59 .
drwxr-xr-x 3 root root 4.0K Jun 24 01:59 ..
-rw-r--r-- 1 root root 2.9M Jun 24 01:59 movies.csv
-rw-r--r-- 1 root root 647M Jun 24 01:59 ratings.csv


In [6]:
movie_path = "/root/.cache/kagglehub/datasets/parasharmanas/movie-recommendation-system/versions/1/movies.csv"
rating_path = "/root/.cache/kagglehub/datasets/parasharmanas/movie-recommendation-system/versions/1/ratings.csv"
user_ratings_df = pd.read_csv(rating_path)
user_ratings_df.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [7]:
# Filter for most active users and most rated movies
top_users = user_ratings_df['userId'].value_counts().head(500).index  # top 500 active users
top_movies = user_ratings_df['movieId'].value_counts().head(500).index  # top 500 popular movies

filtered_ratings = user_ratings_df[user_ratings_df['userId'].isin(top_users) & user_ratings_df['movieId'].isin(top_movies)]


In [8]:
# Load and filter movie metadata
movie_metadata = pd.read_csv(movie_path)
movie_metadata = movie_metadata[movie_metadata['movieId'].isin(filtered_ratings['movieId'].unique())]
movie_metadata.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller


In [9]:
# Merge datasets
movie_data = filtered_ratings.merge(movie_metadata, on='movieId')


In [19]:
movie_data.head()

,userId,movieId,rating,timestamp,title,genres
0,548,1,4.5,1431644949,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,548,2,4.0,1431645039,Jumanji (1995),Adventure|Children|Fantasy
2,548,6,4.0,1431645064,Heat (1995),Action|Crime|Thriller
3,548,10,3.5,1431644914,GoldenEye (1995),Action|Adventure|Thriller
4,548,19,2.5,1431645045,Ace Ventura: When Nature Calls (1995),Comedy


In [10]:
# Create user-item matrix (pivot table)
user_item_matrix = movie_data.pivot(index='userId', columns='movieId', values='rating').fillna(0)
user_item_matrix

movieId,1,2,3,5,6,7,10,11,16,17,...,111759,112556,112852,115713,116797,122882,122886,122904,134130,134853
userId,,,,,,,,,,,,,,,,,,,,,
548,4.5,4.0,0.0,0.0,4.0,0.0,3.5,0.0,0.0,0.0,...,4.0,4.0,4.0,3.5,3.5,4.5,0.0,0.0,0.0,0.0
626,4.5,4.0,0.0,0.0,0.0,4.5,4.0,4.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
847,4.0,0.0,0.0,0.0,4.5,3.0,2.5,0.0,2.5,4.5,...,0.0,4.0,0.0,3.0,0.0,4.5,3.0,0.0,3.0,4.0
997,4.5,3.5,0.0,3.0,4.5,0.0,3.0,0.0,3.5,0.0,...,4.5,4.5,4.5,4.0,3.5,3.5,4.5,5.0,4.0,4.0
1748,4.5,3.0,2.5,4.0,4.0,3.5,4.5,4.5,3.5,2.0,...,4.0,4.0,3.0,2.0,4.5,0.0,3.5,0.0,5.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160747,4.0,3.0,0.0,0.0,4.0,2.0,1.5,2.0,3.5,4.5,...,3.5,3.0,4.0,4.0,0.0,3.5,4.0,0.0,3.5,3.0
160951,4.0,3.0,3.0,3.0,4.0,3.0,0.0,0.0,3.5,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
162047,3.0,2.5,1.5,0.0,4.5,2.0,3.5,0.0,5.0,0.0,...,4.0,4.5,2.0,3.5,3.5,3.0,0.0,0.0,4.0,3.0


In [11]:
# Transpose the matrix to make it movie-item matrix for item-based filtering
movie_user_matrix = user_item_matrix.T

In [12]:
# Store movieId to title mapping
movie_names = movie_metadata.set_index('movieId')

In [13]:
# Define a KNN model on cosine similarity
cf_knn_model= NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=10, n_jobs=-1)


# Fitting the model on our matrix
cf_knn_model.fit(user_item_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1, n_neighbors=10)

In [17]:
# Recommender function using Item-based Collaborative Filtering with KNN
def movie_recommender_engine(movie_name, matrix, cf_model, n_recs):
    # Step 1: Match the input movie name to the closest title in the dataset using fuzzy matching
    movie_id = process.extractOne(movie_name, movie_names['title'])[2]

    # Step 2: Get the vector for the selected movie from the movie-user matrix
    # Reshape it to match the input shape required by kneighbors
    # Retrieve n_recs + 1 nearest neighbors (including the movie itself)
    distances, indices = cf_model.kneighbors(matrix.loc[movie_id].values.reshape(1, -1), n_neighbors=n_recs + 1)

    # Step 3: Combine the neighbor indices and distances into a list of tuples
    # Skip the first one as it is the movie itself (self-match)
    movie_rec_ids = list(zip(indices.squeeze().tolist(), distances.squeeze().tolist()))[1:]

    # Step 4: Prepare a list to store the recommended movies
    cf_recs = []
    for i in movie_rec_ids:
        rec_movie_id = matrix.index[i[0]]  # Get the actual movieId from the matrix index
        title = movie_names.loc[rec_movie_id]['title']  # Look up the title using the movieId
        cf_recs.append({'Title': title, 'Distance': i[1]})  # Store title and similarity distance

    # Step 5: Return the recommendations as a pandas DataFrame
    return pd.DataFrame(cf_recs, index=range(1, n_recs + 1))


In [18]:
# Get recommendations
n_recs = 5
movie_recommender_engine("Batman", movie_user_matrix, cf_knn_model, n_recs)

,Title,Distance
1,Batman Returns (1992),0.134770
2,Batman (1989),0.146871
3,Spider-Man (2002),0.149520
4,Independence Day (a.k.a. ID4) (1996),0.150525
5,"Matrix, The (1999)",0.152510
